In [12]:
import os
from google.colab import drive
import torch
import zipfile
from pathlib import Path

In [ ]:
#os.listdir('/content')
!cp -f '/content/.env' /content/quantpilot/

**check for GPU and mount the drive-->copy conetent from there to colab disc**

In [ ]:

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: False


In [ ]:
!pkill -f "drivefs" && exit
drive.mount('/content/drive')

^C
Mounted at /content/drive


In [ ]:
# Update this path to your ZIP's actual location
!cp "/content/drive/MyDrive/quantpilot.zip" /content/quantpilot.zip

drive.flush_and_unmount()

In [ ]:
!rm -rf /content/quantpilot

**Extract the project & inspect files**

In [ ]:

disc_zip_path = "/content/quantpilot.zip"
extract_path = Path("/content/")

with zipfile.ZipFile(disc_zip_path, "r") as z:
    z.extractall(extract_path)

print(*extract_path.iterdir(), sep="\n")

#!cp -r "/content/content/quantpilot_edit/quantpilot" /content/
#!rm -rf /content/content

/content/.config
/content/quantpilot
/content/.env
/content/quantpilot.zip
/content/sample_data


In [ ]:
!find /content/quantpilot -maxdepth 3 \
  \( -name "requirements-colab.txt" -o -name "pyproject.toml" -o -name "pytest.ini" \)

/content/quantpilot/requirements-colab.txt


**Make changes to the qunatpilot file and zip it back to drive(this step only when needed)**

Zip the chbages at saem level

In [13]:
# Define your target file
input_path = Path("/content/quantpilot")

# The zip archive path at the same level (e.g., my_folder.zip)
zip_path = input_path.with_suffix(".zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    if input_path.is_dir():
        # Loop through all files and subfolders inside the target folder
        for file in input_path.rglob("*"):
            # arcname maintains the relative structure inside the zip
            zf.write(file, arcname=file.relative_to(input_path.parent))
    else:
        # Fallback for a single file
        zf.write(input_path, arcname=input_path.name)

print(f"Successfully zipped and overwrote archive at: {zip_path}")



Successfully zipped and overwrote archive at: /content/quantpilot.zip


Copy the zip back to drive

In [ ]:
drive.mount('/content/drive')

# Update this path from ZIP's actual location back to drive
!cp "/content/quantpilot.zip" /content/drive/MyDrive/quantpilot.zip

drive.flush_and_unmount()


Mounted at /content/drive


**install colab requirements**

In [ ]:
%cd /content/quantpilot
!pip install -r /content/quantpilot/requirements-colab.txt

/content/quantpilot
INFO: pip is looking at multiple versions of langchain-core to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-core to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 25.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 812.1/812.1 kB 16.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 68.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 49.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 92.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.6/3

In [ ]:
import requests, litellm, langchain_litellm
print(requests.__version__)
print(litellm.__file__)

import importlib.metadata

version = importlib.metadata.version("litellm")
print(version)
#print(litellm.__version__)

#!pip install litellm
#!pip install litellm langchain-litellm
#!pip list

2.34.2
/usr/local/lib/python3.13/dist-packages/litellm/__init__.py
1.106.6


In [ ]:
!grep -nE 'numpy|pandas|datetime' /content/quantpilot/requirements*.txt

/content/quantpilot/requirements-colab.txt:24:numpy==2.2.6
/content/quantpilot/requirements-colab.txt:25:pandas==2.2.3
/content/quantpilot/requirements.txt:19:numpy==2.5.3
/content/quantpilot/requirements.txt:20:pandas==3.0.6
/content/quantpilot/requirements.txt:25:datetime


**refine requirements-colab.txt**

In [ ]:
'''
# Extract the ZIP
!unzip -q /content/quantpilot.zip -d /content/quantpilot_edit

# Edit requirements-colab.txt
%cd /content/quantpilot_edit/quantpilot
'''

'\n# Extract the ZIP\n!unzip -q /content/quantpilot.zip -d /content/quantpilot_edit\n\n# Edit requirements-colab.txt\n%cd /content/quantpilot_edit/quantpilot\n'

In [ ]:
#!zip -qr /content/quantpilot_updated.zip /content/quantpilot_edit/quantpilot

In [ ]:
%cd /content/quantpilot
!python --version
!pip check

/content/quantpilot
Python 3.13.18
ipython 7.34.0 requires jedi, which is not installed.
google-colab 1.0.0 has requirement google-auth==2.49.0, but you have google-auth 2.62.1.
google-colab 1.0.0 has requirement requests==2.32.4, but you have requests 2.34.2.


In [ ]:
#!pip3 install jedi
#!pip install google-auth==2.49.0

In [ ]:
!pip check
!python -c "import torch, transformers, langgraph; print('Imports OK'); print('CUDA:', torch.cuda.is_available())"

ipython 7.34.0 requires jedi, which is not installed.
google-colab 1.0.0 has requirement google-auth==2.49.0, but you have google-auth 2.62.1.
google-colab 1.0.0 has requirement requests==2.32.4, but you have requests 2.34.2.
Imports OK
CUDA: False


### **Regression test**

In [ ]:
%cd /content/quantpilot
#!pytest tests/test_workflow.py -q
!pytest tests/test_output_guardrail.py -q

/content/quantpilot
...                                                                      [100%]
3 passed in 24.87s


In [ ]:
%cd /content/quantpilot
!cat src/llm/__init__.py

/content/quantpilot
from .factory import (
    get_model,
    invoke_with_fallback,
)

__all__ = [
    "get_model",
    "invoke_with_fallback",
]

**live gpu smoke test--->loads primary Qwen**

In [ ]:
%cd /content/quantpilot

import torch
from src.llm.factory import get_model

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

model = get_model("reasoning")


/content/quantpilot
CUDA available: True
GPU: Tesla T4


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (5 outputs left out)

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

[HF] Base model ready: Qwen/Qwen3-8B


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Model response: QuantPilot GPU test passed.


In [ ]:
response = model.invoke("In one sentence, explain trailing P/E.")
print("Model response:", response.content)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Model response: Trailing P/E is the ratio of a company's current market price per share to its earnings per share over the past twelve months.


**complete test_workflow run**

In [ ]:
get_model("reasoning")
#!python -m tests.test_workflow

ChatHuggingFace(metadata={'lc_versions': {'langchain-core': '1.6.10', 'langchain': '1.4.3'}}, llm=HuggingFacePipeline(metadata={'lc_versions': {'langchain-core': '1.6.10', 'langchain': '1.4.3'}}, pipeline=TextGenerationPipeline: {'model': 'Qwen3ForCausalLM', 'dtype': 'bfloat16', 'device': 'cuda', 'input_modalities': 'text', 'output_modalities': ('text',)}, model_id='Qwen/Qwen3-8B'), tokenizer=Qwen2Tokenizer(name_or_path='Qwen/Qwen3-8B', vocab_size=151643, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|o

In [ ]:
%cd /content/quantpilot
import torch
print("GPU:", torch.cuda.get_device_name(0))
from src.graph.workflow import compiled_quantpilot_graph
print("Workflow import OK")

/content/quantpilot


AssertionError: Torch not compiled with CUDA enabled

bottlneck resolution

In [ ]:
from transformers import AutoTokenizer

model_id = "Qwen/Qwen3-8B"
print("Loading tokenizer...", flush=True)
tokenizer = AutoTokenizer.from_pretrained(model_id)
print("Tokenizer loaded:", tokenizer.__class__.__name__, flush=True)

Loading tokenizer...
Tokenizer loaded: Qwen2Tokenizer


In [ ]:
from src.llm.factory import get_model

print("Starting isolated factory load...", flush=True)
model = get_model("reasoning")
print("Factory load completed.", flush=True)

print("Testing invocation...", flush=True)
response = model.invoke("Reply with exactly: OK")
print("Response:", response.content, flush=True)

Starting isolated factory load...


OSError: No usable LiteLLM deployments for 'reasoning'. Check provider API keys: ['GROQ_API_KEY', 'CLOUDFLARE_API_KEY', 'CLOUDFLARE_API_KEY', 'NVIDIA_API_KEY', 'CLOUDFLARE_API_KEY', 'CLOUDFLARE_API_KEY', 'GROQ_API_KEY', 'CLOUDFLARE_API_KEY', 'KILOCODE_API_KEY', 'GOOGLE_API_KEY']

In [ ]:
%cd /content/quantpilot

import importlib.util
import nest_asyncio
nest_asyncio.apply()

spec = importlib.util.spec_from_file_location(
    "test_workflow",
    "/content/quantpilot/tests/test_workflow.py",
)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

await module.main()

In [ ]:
import sys
print(sys.__stderr__)
print(sys.__stderr__.fileno())

<_io.TextIOWrapper name='<stderr>' mode='w' encoding='utf-8'>
2


In [ ]:
%cd /content/quantpilot
!ls -la tests

/content/quantpilot
total 140
drwxr-xr-x  3 root root  4096 Oct 25 22:24 .
drwxr-xr-x 12 root root  4096 Oct 25 22:24 ..
-rw-r--r--  1 root root     0 Oct 25 22:24 __init__.py
drwxr-xr-x  2 root root  4096 Oct 25 22:24 __pycache__
-rw-r--r--  1 root root  5983 Oct 25 22:24 test_devils_advocate.py
-rw-r--r--  1 root root  6004 Oct 25 22:24 test_evidence_adjudicator.py
-rw-r--r--  1 root root  7786 Oct 25 22:24 test_evidence_builder.py
-rw-r--r--  1 root root   538 Oct 25 22:24 test_evidence.py
-rw-r--r--  1 root root  4433 Oct 25 22:24 test_final_report.py
-rw-r--r--  1 root root   326 Oct 25 22:24 test_financials.py
-rw-r--r--  1 root root   748 Oct 25 22:24 test_input_guardrail.py
-rw-r--r--  1 root root  7476 Oct 25 22:24 test_llm_factory.py
-rw-r--r--  1 root root   281 Oct 25 22:24 test_market.py
-rw-r--r--  1 root root  1663 Oct 25 22:24 test_output_guardrail.py
-rw-r--r--  1 root root  3429 Oct 25 22:24 test_pii_guardrail.py
-rw-r--r--  1 root root  1609 Oct 25 22:24 test_quant.p

In [ ]:
import sys
sys.path.insert(0, "/content/quantpilot")

from tests.test_workflow import main
await main()

ModuleNotFoundError: No module named 'tests.test_workflow'

In [ ]:
import tests
print(tests.__file__)

/usr/local/lib/python3.13/dist-packages/tests/__init__.py


**LLM o/p debug**

In [ ]:
import regex as re

In [ ]:
str1= '''
LLM RESPONSE repr: '\n\n```json\n{\n  "task_types": ["financial_analysis", "market_analysis"],\n  "entities": ["Apple", "AAPL"],
\n  "research_questions": [\n    "What is Apple\'s current valuation?",\n    "What is the current market trend for Apple?"\n  ],
\n  "date_status": "none",\n  "start_date": null,\n  "end_date_exclusive": null\n}\n```
'''

str2 ='''
 'Okay, let\'s tackle this query. The user wants to compare Apple\'s valuation and market trends. First, I need to figure out the
  task types. Valuation is part of financial analysis, and market trends relate to market analysis. Since they\'re asking to
  compare these two aspects, the task_types should include both financial_analysis and market_analysis, and also the comparison
  between them.\n\nNext, the entities. The ticker given is AAPL, which is Apple Inc. So entities will be "Apple" and
  "AAPL".\n\nFor research questions, they might want to know how Apple\'s valuation metrics like P/E ratio or market cap compare
  over a period, and how the market trends, maybe stock price movements or industry performance, relate. So questions like
  comparing valuation metrics with historical data and analyzing stock price trends against the market.\n\nDates:
  The user didn\'t specify a time frame, so the default start is 2025-01-01 and end is 2025-01-08 exclusive.
  Since they didn\'t mention dates, date_status is "none", but the defaults are provided. Wait, the rules say if no time period
  is specified, use "none". The user didn\'t give dates, so even though defaults are there, the status should be none. The start
  and end dates from the defaults are included as per the problem\'s note to use supplied defaults when no dates are in the query.
  So start_date is 2025-01-01 and end_date_exclusive is 2025-01-08. Date_status remains "none" because the user didn\'t specify
  any dates themselves.\n\nCheck for any ambiguities. The query doesn\'t mention quarters or fiscal periods, so no ambiguity
  there. The dates are defaults, so no issues. The task types are correctly identified. Entities are straightforward.
  Research questions need to be specific. Maybe "How does Apple\'s current valuation compare to historical metrics?" and
  "What are the recent market trends affecting Apple\'s stock performance?" That should cover both aspects and the comparison.
  \n</think>\n\n{\n  "task_types": ["market_analysis", "financial_analysis", "comparison"],\n  "entities": ["Apple", "AAPL"],\n
  "research_questions": [\n    "How does Apple\'s current valuation (P/E ratio, market cap) compare to historical averages?",
  \n    "What are the key market trends impacting AAPL\'s stock performance over the specified period?"\n  ],\n  "date_status":
  "none",\n  "start_date": "2025-01-01",\n  "end_date_exclusive": "2025-01-08"\n}'
'''

span_str1 =re.search(r"\{[\s\S]*?\}", str1).group(0)
print(span_str1)


{
  "task_types": ["financial_analysis", "market_analysis"],
  "entities": ["Apple", "AAPL"],

  "research_questions": [
    "What is Apple's current valuation?",
    "What is the current market trend for Apple?"
  ],

  "date_status": "none",
  "start_date": null,
  "end_date_exclusive": null
}


In [ ]:
import litellm
import os

#litellm.set_verbose = True

os.environ['LITELLM_LOG'] = 'DEBUG'

ModuleNotFoundError: No module named 'litellm'

### **node wise/agent function level reponse debug**

**QuantstaePilot input**

In [ ]:
%cd /content/quantpilot
from src.graph.state import QuantPilotState
from datetime import date
from pprint import pprint

initial_state=QuantPilotState({
    #"user_query": "Analyze Amazon's historical volatility and maximum drawdown.",
    #"user_query": "Calculate and assess NVIDIA's historical returns.",
    #"user_query": "Analyze Apple's recent market trend and price movement.",
    #"user_query": "Analyze Apple's historical volatility.",#case-B(a); without input fin_metruc:- caseB(b)
    #"user_query":"Analyze apple's valuation & historical volatility.",#case 1(with metrics)
    #"user_query": "Analyze apple's valuation and fiftyDayAverage.",
    #"user_query": "Compare Apple's valuation, and market trend for 2nd fortnight of january, 2025",#w/o input fin_metrics;case(C)
    "user_query": "Compare Apple's returns and valuation, beta and market trend between 9th jan to 17th jan, 2025.",# this
    #query will qulaify #for fina;_node in 2nd re-planne run (because of task-wise
    # incremental tool exceution in 2nd run(we cahnged the coomon tool union set to task-
    # specific tool exceution adjudicator check).
    #also this query shall yeid 3 task_types:-'compare','financial'(valuation)&
    # qualification_analysis'
    "ticker": "AAPL",
    "start": date(2025, 1, 1),
    "end": date(2025, 1, 8),
    "financial_metrics": ["trailingPE", "marketCap",],
})

/content/quantpilot


**analyze_query**

In [ ]:
from src.agents.query_analyzer import analyze_query_with_context

result = analyze_query_with_context(
    initial_state["user_query"],
    initial_state["ticker"],
    initial_state["start"],
    initial_state["end"],
)

research_query, start, end = result

print("ResearchQuery:", research_query.model_dump())
print("Effective dates:", start, end)



[LLM Gateway] Role: reasoning | Primary model: groq:qwen/qwen3.8-27b | Priority: 1
LiteLLM deployment used:,reasoning-1-0
content(repr):-'{\n  "task_types": [\n    "market_analysis",\n    "financial_analysis",\n    "quantitative_analysis",\n    "comparison"\n  ],\n  "entities": [\n    "Apple",\n    "AAPL"\n  ],\n  "research_questions": [\n    "What were Apple\'s returns between 2025-01-09 and 2025-01-17?",\n    "How did Apple\'s valuation metrics change between 2025-01-09 and 2025-01-17?",\n    "What was the calculated or reported Beta for AAPL during the period 2025-01-09 to 2025-01-17?",\n    "What was the market trend for AAPL between 2025-01-09 and 2025-01-17?"\n  ],\n  "date_status": "resolved",\n  "start_date": "2025-01-09",\n  "end_date_exclusive": "2025-01-18"\n}'
ResearchQuery: {'original_query': "Compare Apple's returns and valuation, beta and market trend between 9th jan to 17th jan, 2025.", 'task_types': ['market_analysis', 'financial_analysis', 'quantitative_analysis', 'co

In [ ]:
initial_state["research_query"] = research_query
initial_state["start"] = start
initial_state["end"] = end

**reseracher_planner**

In [ ]:
from src.schemas.query import ResearchQuery
from src.agents.research_planner import create_research_plan
from pprint import pprint

rq = ResearchQuery.model_validate(initial_state["research_query"])
plan = create_research_plan(rq)

print("=== RESEARCH PLANNER OUTPUT ===")
pprint(plan.model_dump())

=== RESEARCH PLANNER OUTPUT ===
{'query': "Compare Apple's returns and valuation, beta and market trend "
          'between 9th jan to 17th jan, 2025.',
 'research_questions': ["What were Apple's returns between 2025-01-09 and "
                        '2025-01-17?',
                        "How did Apple's valuation metrics change between "
                        '2025-01-09 and 2025-01-17?',
                        'What was the calculated or reported Beta for AAPL '
                        'during the period 2025-01-09 to 2025-01-17?',
                        'What was the market trend for AAPL between 2025-01-09 '
                        'and 2025-01-17?'],
 'steps': [{'objective': 'Collect relevant historical market price data.',
            'required': True,
            'step_id': 1,
            'task_type': 'market_analysis',
            'tool_category': 'market'},
           {'objective': 'Collect relevant company financial metrics.',
            'required': True,
           

In [ ]:
initial_state["research_plan"] = plan

**resaerch_exceutor**

In [ ]:
from src.schemas.research import ResearchPlan  # adjust to its actual module
from src.agents.research_executor import execute_research_plan
from pprint import pprint

plan = ResearchPlan.model_validate(plan.model_dump() if hasattr(plan, "model_dump") else plan)
#to cover in either case if plan is pydantic object or dict

results = await execute_research_plan(
    research_plan=plan,
    ticker=initial_state["ticker"],
    start=initial_state["start"],
    end=initial_state["end"],
    financial_metrics=initial_state.get("financial_metrics"),
)

pprint(results)

LiteLLM deployment used:,reasoning-1-2
[{'result': {'bars': [{'adj_close': 234.9650421142578,
                       'close': 236.85000610351562,
                       'date': '2025-01-10',
                       'high': 240.16000366210938,
                       'low': 233.0,
                       'open': 240.00999450683594,
                       'volume': 61710900},
                      {'adj_close': 232.53453063964844,
                       'close': 234.39999389648438,
                       'date': '2025-01-13',
                       'high': 234.6699981689453,
                       'low': 229.72000122070312,
                       'open': 233.52999877929688,
                       'volume': 49630700},
                      {'adj_close': 231.42344665527344,
                       'close': 233.27999877929688,
                       'date': '2025-01-14',
                       'high': 236.1199951171875,
                       'low': 232.47000122070312,
                       'o

In [ ]:
#print(results)

In [ ]:
'''
source = results[3]["result"]["sources"][0]
print(source.keys())
print("SUMMARY:", source.get("synthesized_summary", "")[:200])
print("SELECTED:", (
    source.get("synthesized_summary")
    or source.get("content", "")
)[:200])
'''

'\nsource = results[3]["result"]["sources"][0]\nprint(source.keys())\nprint("SUMMARY:", source.get("synthesized_summary", "")[:200])\nprint("SELECTED:", (\n    source.get("synthesized_summary")\n    or source.get("content", "")\n)[:200])\n'

In [ ]:
initial_state["research_results"] = results

**evidence_builder**

In [ ]:
from src.agents.evidence_builder import build_evidence_ledger

ledger = build_evidence_ledger(initial_state["research_results"])

print("\n" + "=" * 70)
print("EVIDENCE BUILDER — ISOLATED OUTPUT")
print("=" * 70)
print(f"Total evidence items: {len(ledger.items)}")

for i, item in enumerate(ledger.items, 1):
    print(f"\n--- Evidence {i} ---")
    print(f"Claim       : {item.claim}")
    print(f"Source type : {item.source_type}")
    print(f"Source      : {item.source}")
    print(f"Task type   : {item.task_type}")
    print(f"Tool category: {item.tool_category}")
    print(f"Status      : {item.status}")
    print(f"Confidence  : {item.confidence}")
    print(f"Evidence    : {item.evidence}")

print("\n" + "=" * 70)

{}
task_types:market_analysis--------------------
tool_category:market--------------------
source:historical_prices--------------------
evidence:6 price observations were collected.--------------------
None
[LLM Gateway] Role: judge | Primary model: groq:qwen/qwen3.8-27b | Priority: 1
LiteLLM deployment used:,judge-1-2
repr(contnet)-->'{\n  "status": "PARTIALLY_SUPPORTED",\n  "confidence": 0.6\n}'
task_types:financial_analysis--------------------
tool_category:financial--------------------
source:financial_metrics--------------------
evidence:trailingPE = 39.523388.--------------------
None
LiteLLM deployment used:,judge-1-2
repr(contnet)-->'{\n  "status": "SUPPORTED",\n  "confidence": 1.0\n}'
task_types:financial_analysis--------------------
tool_category:financial--------------------
source:financial_metrics--------------------
evidence:marketCap = 5157762367488.--------------------
None
LiteLLM deployment used:,judge-1-2
repr(contnet)-->'{\n  "status": "SUPPORTED",\n  "confidence": 

In [ ]:
initial_state['evidence'] = ledger

**thesis_generator**

In [ ]:
# ============================================================
# THESIS NODE — ISOLATED LLM OUTPUT ANALYSIS
# ============================================================

from src.agents.thesis_generator import generate_thesis

# Reuse the EvidenceLedger produced by your previous isolated
# evidence_builder test; do not rebuild or rerun the workflow.
thesis_result = generate_thesis(
    user_query=initial_state["user_query"],
    evidence= initial_state['evidence'],
)

print("\n" + "=" * 70)
print("THESIS NODE — ISOLATED OUTPUT")
print("=" * 70)

print("\nUser query:")
print(initial_state["user_query"])

print("\nGenerated thesis:")
print(thesis_result.thesis)

print("\nConfidence:")
print(thesis_result.confidence)

print("\nSupporting evidence:")
for item in thesis_result.supporting_evidence:
    print("-", item)

print("\nContradicting evidence:")
for item in thesis_result.contradicting_evidence:
    print("-", item)

print("\n" + "=" * 70)

LiteLLM deployment used:,judge-1-1
Thesis raw response: '{"thesis":"There is no direct evidence to analyze Amazon\'s historical volatility and maximum drawdown, as the provided evidence pertains to AAPL (Apple) and not Amazon. However, if we were to analyze AAPL, the data suggests a measured annualized volatility of 14.3906% and a maximum drawdown of -1.1388% over the analyzed period.", "confidence":0.0}'

THESIS NODE — ISOLATED OUTPUT

User query:
Analyze Amazon's historical volatility and maximum drawdown.

Generated thesis:
There is no direct evidence to analyze Amazon's historical volatility and maximum drawdown, as the provided evidence pertains to AAPL (Apple) and not Amazon. However, if we were to analyze AAPL, the data suggests a measured annualized volatility of 14.3906% and a maximum drawdown of -1.1388% over the analyzed period.

Confidence:
0.0

Supporting evidence:
- Total return = -0.6725%.
- Annualized volatility = 14.3906%.
- Maximum drawdown = -1.1388%.

Contradicting 

In [ ]:
initial_state['thesis'] = thesis_result

**devils' advocate**

In [ ]:
# ============================================================
# DEVIL'S ADVOCATE — ISOLATED LLM OUTPUT ANALYSIS
# ============================================================

from src.agents.devils_advocate import critique_thesis

da_result = critique_thesis(
    thesis=initial_state['thesis'],
    evidence=initial_state['evidence'],
    research_query= initial_state['research_query'],
    replan_count=0,
    replan_financial_metrics=None,
)

print("\n" + "=" * 70)
print("DEVIL'S ADVOCATE — ISOLATED OUTPUT")
print("=" * 70)

print("\nOriginal thesis:")
print(thesis_result.thesis)

print("\nThesis confidence:")
print(thesis_result.confidence)

print("\nGenerated critique:")
print(da_result.critique)

print("\nCritique confidence:")
print(da_result.confidence)

print("\nDeterministically challenged evidence:")
for item in da_result.challenged_evidence:
    print("-", item)

print("\nDeterministically identified missing evidence:")
for item in da_result.missing_evidence:
    print("-", item)

print("\n" + "=" * 70)

LiteLLM deployment used:,judge-1-2
Devil's Advocate raw response: '{\n  "critique": "1. **Insufficient Data Sample**: The thesis relies on a very small sample size of only 3-4 price observations to derive volatility and return metrics. This limited data set severely undermines the statistical reliability of these metrics, making it difficult to draw meaningful conclusions about AAPL\'s historical performance. 2. **Lack of Financial and Research Evidence**: The analysis is missing crucial financial and research evidence necessary for a comprehensive financial and quantitative analysis. This omission weakens the thesis by failing to provide a broader context for AAPL\'s valuation and performance. 3. **Unreliable Return Series Measurement**: The return series measurement is only partially supported and has low confidence. This indicates significant uncertainty in the return data, which is a critical component of the thesis. Without reliable return data, the conclusions about AAPL\'s perfo

In [ ]:
initial_state['critique'] = da_result

**evidence adjudicator**

In [ ]:
# ============================================================
# EVIDENCE ADJUDICATOR — ISOLATED LLM OUTPUT ANALYSIS
# ============================================================

from src.agents.evidence_adjudicator import adjudicate_evidence

adjudication_result = adjudicate_evidence(
    thesis=initial_state['thesis'],
    critique= initial_state['critique'],
    evidence= initial_state['evidence'],
)

print("\n" + "=" * 70)
print("EVIDENCE ADJUDICATOR — ISOLATED OUTPUT")
print("=" * 70)

print("\nThesis:")
print(thesis_result.thesis)

print("\nThesis confidence:")
print(thesis_result.confidence)

print("\nDevil's Advocate critique:")
print(da_result.critique)

print("\nCritique confidence:")
print(da_result.confidence)

print("\nDecision:")
print(adjudication_result.decision)

print("\nAdjudication rationale:")
print(adjudication_result.rationale)

print("\nAdjudication confidence:")
print(adjudication_result.confidence)

print("\nEvidence debt:")
for item in adjudication_result.evidence_debt:
    print("-", item)

print("\nMissing tools:")
for item in adjudication_result.missing_tools:
    print("-", item)

print('\nQuality score:')
print(adjudication_result.quality_score)

print("\n Quality band")
print(adjudication_result.quality_band)

print("\n" + "=" * 70)

LiteLLM deployment used:,judge-1-0
Adjudicator raw response: '{\n  "rationale": "The decision to FAIL is justified by significant evidence gaps and statistical limitations. While valuation metrics (P/E, Market Cap) are supported with high confidence, the quantitative analysis relies on only 3-4 price observations, rendering volatility and return metrics statistically unreliable. The evidence ledger explicitly flags missing financial and research evidence for both financial and quantitative analysis tasks. Furthermore, the return series is marked as partially supported with low confidence, undermining the thesis\'s performance claims. The critique correctly identifies that without broader historical context and robust data samples, the thesis cannot be validated.",\n  "confidence": 0.95\n}'

EVIDENCE ADJUDICATOR — ISOLATED OUTPUT

Thesis:
Apple (AAPL) exhibits a high valuation with a trailing P/E of approximately 39.9 and a market capitalization of roughly $5.2 trillion. Over the limite

In [ ]:
initial_state['adjudication'] = adjudication_result

**replan researcher**

In [ ]:
from src.agents.replan_researcher import replan_research
from src.schemas.query import ResearchQuery
from src.schemas.evidence import EvidenceLedger
from src.schemas.adjudication import Adjudication

# Reuse the same query, evidence ledger, and adjudication
# from your latest isolated workflow run.
replan_plan, replan_financial_metrics = replan_research(
    research_query= initial_state['research_query'],
    evidence= initial_state['evidence'],
    adjudication= initial_state['adjudication'],
)

print("\n" + "=" * 65)
print("REPLAN RESEARCHER — ISOLATED OUTPUT")
print("=" * 65)

print("\nReplan query:")
print(replan_plan.query)

print("\nReplan financial metrics:")
print(replan_financial_metrics)

print("\nReplan steps:")
for step in replan_plan.steps:
    print(
        f"- Step {step.step_id} | "
        f"task_type={step.task_type} | "
        f"tool_category={step.tool_category} | "
        f"required={step.required} | "
        f"objective={step.objective}"
    )

print("\nResearch questions:")
for question in replan_plan.research_questions:
    print(f"- {question}")

print("=" * 65)


REPLAN RESEARCHER — ISOLATED OUTPUT

Replan query:
Analyze apple's valuation & historical volatility.

Replan financial metrics:
[]

Replan steps:
- Step 1 | task_type=financial_analysis | tool_category=financial | required=False | objective=Collect the missing financial evidence.
- Step 2 | task_type=financial_analysis | tool_category=research | required=True | objective=Gather the missing external research evidence.
- Step 3 | task_type=quantitative_analysis | tool_category=financial | required=False | objective=Collect the missing financial evidence.
- Step 4 | task_type=quantitative_analysis | tool_category=quantitative | required=True | objective=Perform the missing quantitative analysis.

Research questions:
- What is the current valuation of Apple (AAPL)?
- What is the historical volatility of Apple (AAPL)?


In [ ]:
initial_state['replan_plan'] = replan_plan
initial_state['replan_financial_metrics'] = replan_financial_metrics
initial_state['replan_count'] = initial_state.get("replan_count", 0) +1

In [ ]:
#initial_state

**research_exceutor_replan run**

In [ ]:
from src.agents.research_executor import execute_research_plan

print("\n" + "=" * 65)
print("RESEARCH EXECUTOR — REPLAN RUN")
print("=" * 65)


replan_results = await execute_research_plan(
    research_plan=initial_state["replan_plan"],
    ticker=initial_state["ticker"],
    start=initial_state["start"],
    end=initial_state["end"],
    financial_metrics=initial_state.get("financial_metrics"),
    replan_financial_metrics=initial_state.get("replan_financial_metrics"),
    existing_results=initial_state.get("research_results", []),
    replan_count=initial_state.get("replan_count", 0),
)

for i, item in enumerate(replan_results, 1):
    print(f"\nResult {i}")
    print(f"Tool category: {item.get('tool_category')}")
    result = item.get("result")

    if result is None:
        print("Result: None (intentional no-operation placeholder)")
    else:
        print(f"Result type: {type(result).__name__}")
        #print(f"Result: {str(result)[:1000]}")
        print(f"Result:-{result}")

print("\n" + "=" * 65)


RESEARCH EXECUTOR — REPLAN RUN
LiteLLM deployment used:,reasoning-1-0

Result 1
Tool category: financial
Result type: dict
Result:-{'ticker': 'AAPL', 'metrics': [{'name': 'trailingPE', 'period': None, 'value': 39.887398}, {'name': 'marketCap', 'period': None, 'value': 5205264957440}]}

Result 2
Tool category: research
Result type: dict
Result:-{'query': "Analyze apple's valuation & historical volatility.", 'sources': [{'title': 'Apple Inc. (AAPL) - Historical Volatility (Close-to-Close) (30-Day)', 'url': 'https://www.alphaquery.com/stock/AAPL/volatility-option-statistics/30-day/historical-volatility', 'content': 'Interested in downloading volatility data? Check out our premium data product, VolVue. Query and download extensive historical data in CSV, Excel, JSON, or XML format, via web user interface or API.\n\n# Apple Inc. (AAPL)\n\nLast Closing Price: 342.37 (2026-11-06)\n\n Profile & Key Metrics\n Price Chart\n Financials & Fundamentals\n Industry/Sector/Market Percentiles\n Earnin

In [ ]:
# Update state with the replan executor output
initial_state["research_results"] = (initial_state.get("research_results", []) + replan_results)

**evidence_builder_replan run**

In [ ]:
from src.agents.evidence_builder import build_evidence_ledger

print("\n" + "=" * 65)
print("EVIDENCE BUILDER — REPLAN RUN")
print("=" * 65)

replan_ledger = build_evidence_ledger(
    initial_state["research_results"],
    initial_state["evidence"]
)

initial_state["evidence"] = replan_ledger

for i, item in enumerate(replan_ledger.items, 1):
    print(f"\nEvidence {i}")
    print(f"Task type     : {item.task_type}")
    print(f"Tool category : {item.tool_category}")
    print(f"Claim         : {item.claim}")
    print(f"Status        : {item.status}")
    print(f"Confidence    : {item.confidence}")
    print(f"Evidence      : {item.evidence[:500]}")

print("\nTotal evidence items:", len(replan_ledger.items))
print("=" * 65)


EVIDENCE BUILDER — REPLAN RUN
{'a8cd429b2d879d13dfd42716a10bfaf6db24d7408f384513c0af8135b25f6937': Evidence(claim='AAPL has a reported trailingPE value.', source_type='financial_data', source='financial_metrics', task_type='financial_analysis', tool_category='financial', step_id=1, evidence='trailingPE = 39.887398.', confidence=0.95, status='SUPPORTED'), '367c6428f3818ebfa1e45793e83c999aa92577a16e341d72017ab7937b5049dc': Evidence(claim='AAPL has a reported marketCap value.', source_type='financial_data', source='financial_metrics', task_type='financial_analysis', tool_category='financial', step_id=1, evidence='marketCap = 5205264957440.', confidence=1.0, status='SUPPORTED'), '70a8a60c8305f119fc8ca025b3af27327c546fd63dd3d8a6ba32905d0667aac2': Evidence(claim='AAPL historical price data was collected.', source_type='market_data', source='historical_prices', task_type='quantitative_analysis', tool_category='market', step_id=2, evidence='4 price observations were collected.', confidence=0.

**thesis generator replan run**

In [ ]:
from src.agents.thesis_generator import generate_thesis

# Generate thesis from the updated replan evidence
replan_thesis = generate_thesis(
    user_query=initial_state["user_query"],
    evidence=initial_state["evidence"],
)

# Update state for the next node
initial_state["thesis"] = replan_thesis

print("\n" + "=" * 70)
print("THESIS GENERATOR — REPLAN RUN")
print("=" * 70)

print("\nUser query:")
print(initial_state["user_query"])

print("\nGenerated thesis:")
print(replan_thesis.thesis)

print("\nConfidence:")
print(replan_thesis.confidence)

print("\nSupporting evidence:")
for item in replan_thesis.supporting_evidence:
    print(f"- {item}")

print("\nContradicting evidence:")
for item in replan_thesis.contradicting_evidence:
    print(f"- {item}")

print("=" * 70)

LiteLLM deployment used:,judge-1-0
Thesis raw response: '{\n  "thesis": "Apple (AAPL) has a market capitalization of approximately $5.21 trillion and a trailing P/E ratio of 39.89. Historical volatility metrics conflict: internal data reports an annualized volatility of 14.39% with a maximum drawdown of -1.14% and a total return of -0.67% over a period with only 3-4 observations, while external 30-day data reports a higher close-to-close historical volatility of 24.95% and an implied volatility mean of 25.93%. The limited sample size of the internal data limits the reliability of its volatility estimate compared to the external 30-day metrics.",\n  "confidence": 0.85\n}'

THESIS GENERATOR — REPLAN RUN

User query:
Analyze apple's valuation & historical volatility.

Generated thesis:
Apple (AAPL) has a market capitalization of approximately $5.21 trillion and a trailing P/E ratio of 39.89. Historical volatility metrics conflict: internal data reports an annualized volatility of 14.39% w

**devil's advocate replan run**

In [ ]:
from src.agents.devils_advocate import critique_thesis

print("\n" + "=" * 70)
print("DEVIL'S ADVOCATE — REPLAN RUN")
print("=" * 70)

# Run against the updated thesis and accumulated evidence
replan_critique = critique_thesis(
    thesis=initial_state["thesis"],
    evidence=initial_state["evidence"],
    research_query=initial_state["research_query"],
    replan_count=initial_state["replan_count"],
)

# Preserve the result for the next node
initial_state["critique"] = replan_critique

print("\nUser query:")
print(initial_state["user_query"])

print("\nReplan count:")
print(initial_state["replan_count"])

print("\nCritique:")
print(replan_critique.critique)

print("\nConfidence:")
print(replan_critique.confidence)

print("\nChallenged evidence:")
for item in replan_critique.challenged_evidence:
    print(f"- {item}")

print("\nMissing evidence:")
for item in replan_critique.missing_evidence:
    print(f"- {item}")

print("=" * 70)


DEVIL'S ADVOCATE — REPLAN RUN
LiteLLM deployment used:,judge-1-0
Devil's Advocate raw response: '{\n  "critique": "The thesis relies on a fundamental methodological flaw: comparing a 3-4 observation internal sample to a 30-day external sample is statistically invalid, rendering the \'conflict\' a comparison of non-equivalent datasets rather than a genuine data discrepancy. The internal volatility (14.39%) is statistically meaningless with only 3 return observations, yet the thesis treats it as a valid counterpoint to the external 24.95% figure. Furthermore, the thesis conflates descriptive statistics with analytical insight; it lists market cap and P/E without linking them to the volatility metrics, leaving the \'so what\' of the valuation-volatility relationship unaddressed. The external data is a single-point snapshot (Nov 6, 2026) from one source (AlphaQuery), lacking independent corroboration or historical context to confirm if 24.95% is anomalous or typical. The thesis fails to d

**evidence_adjudicator replan run**

In [ ]:
from src.agents.evidence_adjudicator import adjudicate_evidence

# Use outputs from the replan Thesis Generator, Devil's Advocate,
# and updated Evidence Builder cells.
replan_adjudication = adjudicate_evidence(
    initial_state["thesis"],
    initial_state["critique"],
    initial_state['evidence']
)

print("\n===== REPLAN EVIDENCE ADJUDICATOR =====")
print("Decision:", replan_adjudication.decision)
print("Rationale:", replan_adjudication.rationale)
print("Confidence:", replan_adjudication.confidence)
print("Evidence debt:", replan_adjudication.evidence_debt)
print("Missing tools:", replan_adjudication.missing_tools)
print("Quality score:", replan_adjudication.quality_score)
print("Quality band:", replan_adjudication.quality_band)
print('='*80)

LiteLLM deployment used:,judge-1-2
Adjudicator raw response: '{\n  "rationale": "The decision to FAIL the thesis is justified by several critical issues identified in the critique and supported by the evidence. The thesis compares a 3-4 observation internal sample to a 30-day external sample, which is statistically invalid. The internal volatility estimate (14.39%) is based on an insufficient sample size (3 return observations), making it statistically meaningless. The thesis treats this small sample estimate as a valid counterpoint to the external 24.95% figure, which is derived from a more robust 30-day dataset. Additionally, the thesis does not adequately link market capitalization and P/E ratio to volatility metrics, failing to provide analytical insight into the relationship between valuation and volatility. The external data, while detailed, is a single-point snapshot from one source (AlphaQuery) and lacks independent corroboration or historical context. The thesis also conflates

In [ ]:
initial_state["adjudication"] = replan_adjudication

**final_report**

In [ ]:
# FINAL REPORT — ISOLATED INCREMENTAL TEST
from src.agents.final_report import generate_final_report

final_report = generate_final_report(
    user_query=initial_state["user_query"],
    thesis=initial_state["thesis"],
    adjudication=initial_state["adjudication"],
    research_plan=initial_state["research_plan"],
    research_results=initial_state["research_results"],
    replan_plan=initial_state.get("replan_plan"),
    financial_metrics=initial_state.get("financial_metrics"),
    replan_financial_metrics=initial_state.get("replan_financial_metrics"),
)

print("\n===== FINAL REPORT — ISOLATED OUTPUT =====")
print("User query:", final_report.user_query)
print("Summary:", final_report.summary)
print("Thesis:", final_report.thesis)
print("Conclusion:", final_report.conclusion)
print("Confidence:", final_report.confidence)
print("Normalized results:", final_report.normalized_results)
print("Limitations:", final_report.limitations)
print('='*80)

LiteLLM deployment used:,reasoning-1-0

===== FINAL REPORT — ISOLATED OUTPUT =====
User query: Analyze apple's valuation & historical volatility.
Summary: Apple (AAPL) has a market capitalization of approximately $5.21 trillion and a trailing P/E ratio of 39.89. Measured internal data from January 2025 (4 price observations) indicates an annualized volatility of 14.39%, a maximum drawdown of -1.14%, and a total return of -0.67%. External research from AlphaQuery (dated November 2026) reports a 30-day close-to-close historical volatility of 24.95% and a mean implied volatility of 25.93%. The internal volatility estimate is based on a limited sample of three return observations, reducing its statistical reliability compared to the external 30-day metrics.
Thesis: Apple (AAPL) has a market capitalization of approximately $5.21 trillion and a trailing P/E ratio of 39.89. Historical volatility metrics conflict: internal data reports an annualized volatility of 14.39% with a maximum drawdown

In [ ]:
initial_state['final_report'] = final_report

**output_guardrail**

In [ ]:
import logging
import json
from copy import deepcopy

from src.guardrails.output import validate_final_report

# Capture logs from the output guardrail
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(name)s | %(message)s",
    force=True
)

print("=" * 100)
print("INCREMENTAL TEST: OUTPUT GUARDRAIL")
print("=" * 100)

# 1. Input from the completed workflow
report_before = deepcopy(initial_state["final_report"])

'''
print("\n[INPUT] Final Report:")
print(json.dumps(
    report_before.model_dump(),
    indent=2,
    default=str
))
'''

# 2. Run ONLY the output guardrail
report_after = validate_final_report(
    deepcopy(report_before)
)

# 3. Display output
print("\n" + "=" * 100)
print("[OUTPUT] Output Guardrail Result")
print("=" * 100)

print("\nValidation Remarks:")
print(repr(report_after.validation_remarks))

print("\nComplete Report After Guardrail:")
print(json.dumps(
    report_after.model_dump(),
    indent=2,
    default=str
))

# 4. Compare report fields
print("\n" + "=" * 100)
print("[COMPARISON] Before vs After")
print("=" * 100)

before_data = report_before.model_dump()
after_data = report_after.model_dump()

for field in before_data:
    if before_data[field] != after_data[field]:
        print(f"\nCHANGED FIELD: {field}")
        print("BEFORE:", repr(before_data[field]))
        print("AFTER: ", repr(after_data[field]))

unchanged = all(
    before_data[field] == after_data[field]
    for field in before_data
    if field != "validation_remarks"
)

print("\nReport content unchanged:", unchanged)

INCREMENTAL TEST: OUTPUT GUARDRAIL
[LLM Gateway] Role: guardrail | Primary model: google_genai:gemini-3.5-flash-lite | Priority: 1
16:53:32 - LiteLLM Router:INFO: router.py:1601 - Routing strategy: simple-shuffle


INFO | LiteLLM Router | Routing strategy: simple-shuffle


16:53:32 - LiteLLM:INFO: utils.py:4706 - 
LiteLLM completion() model= gemini-3.5-flash-lite; provider = gemini


INFO | LiteLLM | 
LiteLLM completion() model= gemini-3.5-flash-lite; provider = gemini


16:53:32 - LiteLLM:INFO: vertex_and_google_ai_studio_gemini.py:1147 - Warning: Setting temperature < 1.0 for Gemini 3 models (gemini-3.5-flash-lite) can cause infinite loops, degraded reasoning performance, and failure on complex tasks. Strongly recommended to use temperature = 1.0 (default).


INFO | LiteLLM | Warning: Setting temperature < 1.0 for Gemini 3 models (gemini-3.5-flash-lite) can cause infinite loops, degraded reasoning performance, and failure on complex tasks. Strongly recommended to use temperature = 1.0 (default).
16:53:32 - LiteLLM:WARNING: vertex_and_google_ai_studio_gemini.py:1152 - DeprecationWarning: `temperature`, `top_p`, and `top_k` continue to function for Gemini 3+ (gemini-3.5-flash-lite) but are planned for removal in a future release. Move sampling guidance into the `system` instructions instead.
WARNING | LiteLLM | DeprecationWarning: `temperature`, `top_p`, and `top_k` continue to function for Gemini 3+ (gemini-3.5-flash-lite) but are planned for removal in a future release. Move sampling guidance into the `system` instructions instead.


16:53:33 - LiteLLM:INFO: utils.py:1899 - Wrapper: Completed Call, calling success_handler


INFO | LiteLLM | Wrapper: Completed Call, calling success_handler


16:53:33 - LiteLLM Router:INFO: router.py:2777 - litellm.completion(model=gemini/gemini-3.5-flash-lite) 200 OK


INFO | LiteLLM Router | litellm.completion(model=gemini/gemini-3.5-flash-lite) 200 OK
INFO | src.llm.factory | LiteLLM deployment used: guardrail-1-0


LiteLLM deployment used:,guardrail-1-0

[OUTPUT] Output Guardrail Result

Validation Remarks:
'PASS: No material issues found.'

Complete Report After Guardrail:
{
  "user_query": "Analyze apple's valuation & historical volatility.",
  "summary": "Apple (AAPL) has a market capitalization of approximately $5.21 trillion and a trailing P/E ratio of 39.89. Measured internal data from January 2025 (4 price observations) indicates an annualized volatility of 14.39%, a maximum drawdown of -1.14%, and a total return of -0.67%. External research from AlphaQuery (dated November 2026) reports a 30-day close-to-close historical volatility of 24.95% and a mean implied volatility of 25.93%. The internal volatility estimate is based on a limited sample of three return observations, reducing its statistical reliability compared to the external 30-day metrics.",
  "thesis": "Apple (AAPL) has a market capitalization of approximately $5.21 trillion and a trailing P/E ratio of 39.89. Historical volatilit

**Smoke test(for api LLM initialization and invokation test)**

In [ ]:
%cd /content/quantpilot/
import os
from dotenv import load_dotenv
load_dotenv()

/content/quantpilot


True

In [ ]:

os.getcwd()
#!cp -r '/content/.env' /content/quantpilot

'/content/quantpilot'

In [ ]:
#showed the crednetials in .env
#with open(f"{os.getcwd()}/.env", 'r') as file:
#    print(file.read())

In [ ]:
!git ls-files | findstr ".env"

/bin/bash: line 1: findstr: command not found
fatal: not a git repository (or any of the parent directories): .git


In [ ]:
#os.getenv('GOOGLE_API_KEY')

In [ ]:
os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY')
os.environ['NVIDIA_API_KEY'] = os.getenv('NVIDIA_API_KEY')
os.environ['GROQ_API_KEY'] =os.getenv('GROQ_API_KEY')

In [ ]:
#%cd /content/quantpilot
from langchain_core.messages import HumanMessage
from src.llm.litellm_gateway import get_litellm_model

roles = ["reasoning", "judge", "guardrail"]

for role in roles:
    print(f"\n{'='*60}")
    print(f"Testing LiteLLM Router: {role}")

    try:
        print("-"*40)
        model = get_litellm_model(role)
        print('-'*60)

        response = model.invoke([
            HumanMessage(
                content="Reply with exactly: QuantPilot API ready"
            )
        ])

        print("Response type:", type(response).__name__)
        print("Response:", repr(response.content))

        assert response.content, "Empty response received"

        print(f"<PASS> {role}")

    except Exception as exc:
        print(f"<FAIL> {role}")
        print(f"{type(exc).__name__}: {exc}")


Testing LiteLLM Router: reasoning
----------------------------------------
build_model_list_start
build_model_list_ends
post redis cache before router initialization
post router phase
------------------------------------------------------------
Response type: AIMessage
Response: 'QuantPilot API ready'
<PASS> reasoning

Testing LiteLLM Router: judge
----------------------------------------
build_model_list_start
build_model_list_ends
post redis cache before router initialization
post router phase
------------------------------------------------------------
Response type: AIMessage
Response: 'QuantPilot API ready'
<PASS> judge

Testing LiteLLM Router: guardrail
----------------------------------------
build_model_list_start
build_model_list_ends


01:53:45 - LiteLLM:WARNING: vertex_and_google_ai_studio_gemini.py:1152 - DeprecationWarning: `temperature`, `top_p`, and `top_k` continue to function for Gemini 3+ (gemini-3.5-flash-lite) but are planned for removal in a future release. Move sampling guidance into the `system` instructions instead.


post redis cache before router initialization
post router phase
------------------------------------------------------------
Response type: AIMessage
Response: 'QuantPilot API ready'
<PASS> guardrail
